In [ ]:
# 🛠️ Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens datasets transformers plotly
    # transformer_lens pulls a newer torch; Colab's torchaudio was built against the
    # old torch and fails to load (undefined symbol: torch_library_impl). transformers
    # imports it lazily and crashes, though nothing here uses audio. Remove it.
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

> 🧪 **نسخة التمرين.** الخلايا اللي فيها `# TODO` ناقصة عمدًا — املأها بنفسك.
> لو علقت، النسخة المرجعية (`..._reference.ipynb`) فيها الحل كامل.

</div>

<div dir="rtl">

# المرحلة ٢ج: طبقة تانية — التركيب والاستقراء

لحد دلوقتي كل الرؤوس في **طبقة واحدة** بتشتغل بالتوازي. العمق بيفتح حاجة جديدة مفيش طبقة واحدة تقدر تعملها لوحدها: **التركيب** (composition) — رأس يستعمل **ناتج** رأس تاني.

أهم مثال هو **رأس الاستقراء** (induction head): لو شاف `[أ][ب] … [أ]`، بيتوقع **[ب]** — يعني "الحاجة اللي جت بعد [أ] آخر مرة". ده محتاج رأسين متركّبين:
- **رأس الكلمة-السابقة** (L0): كل موضع بيكتب فيه مين التوكن اللي قبله.
- **رأس الاستقراء** (L1): بيدوّر على المرة اللي فاتت ظهر فيها التوكن الحالي، وبياخد اللي بعده.

**ليه طبقتين؟** عشان رأس الاستقراء يلاقي "المرة اللي فاتت"، المفتاح لازم يحمل معلومة التوكن السابق — واللي بيكتبها هو رأس L0. طبقة واحدة ميقدرش.

**حدود التجربة:** هنعلّم الاستقراء أول حاجة على مهمة **صناعية نظيفة** (توكنز عشوائية بتتكرر) عشان الدائرة تبان واضحة، وبعدين نرجع للعربي.

</div>

<div dir="ltr">

# Stage 2c: A second layer — composition & induction

So far all heads sit in **one layer**, working in parallel. Depth unlocks something no single layer can do: **composition** — one head using another head's **output**.

The headline example is the **induction head**: seeing `[A][B] … [A]` it predicts **[B]** — "what came after [A] last time". This needs two composed heads:
- **prev-token head (L0):** each position records which token preceded it.
- **induction head (L1):** finds the previous occurrence of the current token and copies what followed it.

**Why two layers?** For the induction head to find "last time", the key must carry the previous token's identity — which L0 writes. One layer can't.

**Limits:** we teach induction first on a **clean synthetic task** (random tokens that repeat) so the circuit is unmistakable, then return to Arabic.

</div>

<div dir="rtl">

## ١. مهمة التكرار · The repeat task

`tiny.make_repeated_with_gap` بيعمل تتابع: `[BOS، كتلة A، فجوة عشوائية، نفس الكتلة A]`. الفجوة **عشوائية**، فالتكرار بييجي على مسافة مختلفة كل مرة — يعني مفيش قاعدة موضعية ثابتة تنفع؛ الموديل **لازم** يستعمل المحتوى (الاستقراء). بنعرف مكان كل توكن اتنسخ منين (`src`) عشان نقيس الاستقراء بدقة.

</div>

In [ ]:
V, BLOCK, GAP = 30, 12, 12
N_CTX = 1 + 2 * BLOCK + GAP          # fits [BOS, A, gap, A]
STEPS = globals().get("STEPS", 3000)

tokens, src = tiny.make_repeated_with_gap(4, BLOCK, GAP, V, seed=0)
print("one sequence (ids):", tokens[0].tolist())
print("repeats copied from positions:", [int(s) for s in src[0] if s >= 0])

# TODO: build a TWO-layer model (depth is the whole point of this notebook).
model = tiny.make_tiny_model(n_layers=..., n_heads=2, d_vocab=V, n_ctx=N_CTX, d_model=128)  # TODO n_layers

# TODO: train on FRESH variable-gap data each step so the model must GENERALISE
# (memorising fixed sequences would let it cheat positionally). Fill the loop:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
model.train()
for step in range(STEPS):
    batch, _ = tiny.make_repeated_with_gap(64, BLOCK, GAP, V, seed=step)  # fresh each step
    batch = batch.to(tiny.device())
    # TODO: zero grad -> loss = model(batch, return_type="loss") -> backward -> opt.step()
    ...
    if step % max(STEPS // 5, 1) == 0:
        print(f"  step {step:>4}/{STEPS}  loss={float(loss):.3f}")

<div dir="rtl">

## ٢. فين رأس الاستقراء؟ · Where is the induction head?

بنقيس لكل رأس: قد إيه بيبص من توكن متكرر **للموضع اللي بعد** أول ظهور لنفس التوكن (ده تعريف الاستقراء). نتوقع الدرجة العالية في **الطبقة التانية (L1)** — لأنه محتاج رأس L0 تحته.

</div>

In [ ]:
def induction_from_patterns(patterns, src):
    """patterns: list over layers of (B,H,S,S); src[b,t] = first-occurrence index
    of the token repeated at t (else -1). Score a head by how much a repeated
    position t attends to src[b,t]+1 (the token AFTER the first occurrence)."""
    n_layers, n_heads = len(patterns), patterns[0].shape[1]
    B, S = patterns[0].shape[0], patterns[0].shape[-1]
    scores = torch.zeros(n_layers, n_heads)
    # TODO: for each layer L and head h, average patterns[L][b, h, t, src[b,t] + 1]
    # over every repeated position (src[b,t] >= 0 and src[b,t] + 1 < t). Mind the +1!
    ...  # TODO
    return scores

def attention_scores(model, tokens, src):
    _, cache = model.run_with_cache(tokens.to(model.cfg.device))
    pats = [cache["pattern", L].detach().cpu() for L in range(model.cfg.n_layers)]
    return induction_from_patterns(pats, src), pats

In [ ]:
eval_tokens, eval_src = tiny.make_repeated_with_gap(40, BLOCK, GAP, V, seed=99999)
ind, pats = attention_scores(model, eval_tokens, eval_src)
print("induction score [layer, head]:\n", ind.round(decimals=2))
best = divmod(int(ind.flatten().argmax()), model.cfg.n_heads)
print(f"strongest induction head: layer {best[0]}, head {best[1]}  (score {float(ind.max()):.2f})")

<div dir="rtl">

## ٣. الرأسين · The two heads

نعرض على تتابع واحد: خريطة رأس **L0** (المفروض شكل قطري — كل موضع بيبص للي قبله مباشرة = رأس الكلمة-السابقة) وخريطة رأس **L1** (المفروض "خط استقراء" — التوكنز المتكررة بتبص لمواضع أول ظهور+١). المحور أرقام مواضع (مهمة صناعية، مش عربي).

</div>

In [ ]:
seq_i = 0
toks1 = eval_tokens[seq_i].tolist()
labels = [f"{t}" for t in toks1]
L0 = pats[0][seq_i, int(ind[0].argmax())].numpy()
L1 = pats[1][seq_i, best[1]].numpy()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("L0 head — prev-token (diagonal)",
                                    "L1 head — induction (the stripe)"))
fig.add_trace(go.Heatmap(z=L0, colorscale="Blues", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=L1, colorscale="Blues", showscale=False), row=1, col=2)
fig.update_yaxes(autorange="reversed"); fig.update_layout(height=360, width=760,
    title_text="rows = query position, cols = key position")
fig.show()

<div dir="rtl">

## ٤. التركيب (K-composition) · How they compose

رأس L0 بيكتب في كل موضع معلومة عن **التوكن اللي قبله**. رأس L1 بيقرأ المعلومة دي في الـ**مفتاح** (key) بتاعه — فلما يدوّر على "فين ظهر التوكن الحالي قبل كده"، بيلاقي الموضع الصح وياخد اللي بعده. ده اسمه **K-composition**: مخرجات L0 بتغذّي مفاتيح L1. ده بالظبط اللي ميقدرش يحصل في طبقة واحدة.

</div>

<div dir="rtl">

## ٥. بتشتغل على تتابع جديد؟ · Does it fire on a fresh sequence?

الدائرة الحقيقية لازم تشتغل على **تكرار لسه ماشفهوش**. نولّد تتابع جديد (seed مختلف) ونقيس الاستقراء تاني — لو فضل عالي في L1، يبقى اتعلم **الخوارزمية**، مش حفظ.

</div>

In [ ]:
fresh_tokens, fresh_src = tiny.make_repeated_with_gap(40, BLOCK, GAP, V, seed=123456)
fresh_ind, _ = attention_scores(model, fresh_tokens, fresh_src)
print("induction on a fresh sequence [layer, head]:\n", fresh_ind.round(decimals=2))
print(f"L1 induction holds on unseen data: {float(fresh_ind[1].max()):.2f}")

<div dir="rtl">

## ٦. نرجع للعربي · Back to Arabic

المهمة الصناعية وضّحت الدائرة. دلوقتي نرجع لخيطنا: ندرّب موديل **بطبقتين** على نص عربي حقيقي (نفس مُجزّئ mGPT بتاع ٢أ/٢ب)، ونعيد **شجرة التوقعات** على نفس السياقين — نشوف توقعات موديل الطبقتين على العربي (قارن بـ ٢أ و ٢ب). (الاستقراء على نص طبيعي أصعب وأخفت من المهمة الصناعية النظيفة.)

</div>

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import re

MAX_CHARS = globals().get("MAX_CHARS", 500_000)
N_CTX_AR = globals().get("N_CTX_AR", 64)
N_EPOCHS = globals().get("N_EPOCHS", 200)

msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
eg_tweets = load_dataset("amgadhasan/arabic_tweets_dialects", split="train").filter(
    lambda x: x["dialect"] == "EG")

def clean(t):
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[a-zA-Z0-9_@]+", "", t)
    return re.sub(r"[^\s\u0621-\u064A]", "", t)

def collect(rows, key, n):
    out = ""
    for r in rows:
        out += clean(r[key]) + " "
        if len(out) >= n:
            break
    return out[:n]

mgpt = AutoTokenizer.from_pretrained("ai-forever/mGPT")
encode, corpus_ids, id_to_str, VOCAB = tiny.make_compact_encoder(
    mgpt, [collect(msa_stream, "text", MAX_CHARS), collect(eg_tweets, "text", MAX_CHARS)])
ar_all = corpus_ids[0] + corpus_ids[1]

ar_model = tiny.make_tiny_model(n_layers=2, n_heads=2, d_vocab=VOCAB, n_ctx=N_CTX_AR)
ar_losses = tiny.train(ar_model, tiny.make_natural_batches(ar_all, n_ctx=N_CTX_AR),
                       n_epochs=N_EPOCHS, log_every=20)
print("Arabic 2-layer loss:", round(ar_losses[0], 3), "->", round(ar_losses[-1], 3))

In [ ]:
LOW_PROB = 0.10

def next_topk(context_ids, k):
    ids = torch.tensor([context_ids]).to(tiny.device())
    probs = torch.softmax(ar_model(ids)[0, -1], dim=-1)
    vals, idx = torch.topk(probs, k)
    return [(int(i), float(v)) for v, i in zip(vals, idx)]

def build_prediction_tree(context_text, max_depth=2, top_k=2):
    edges, nodes = [], {"root": (context_text, True)}
    frontier, n = [("root", encode(context_text), 0)], 0
    while frontier:
        pkey, ctx, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        for tid, p in next_topk(ctx, top_k):
            n += 1
            label = id_to_str.get(tid, "[UNK]")
            sensible = (label != "[UNK]") and (p >= LOW_PROB)
            ckey = f"{depth}:{n}:{label}"
            nodes[ckey] = (label, sensible)
            edges.append((pkey, ckey, p, depth, sensible))
            frontier.append((ckey, ctx + [tid], depth + 1))
    return edges, nodes

def tree_layout(edges):
    depth_of = {"root": 0}
    for _pk, ck, _p, d, _s in edges:
        depth_of[ck] = d + 1
    levels = {}
    for k, d in depth_of.items():
        levels.setdefault(d, []).append(k)
    return {k: (d, (len(ks) - 1) / 2 - i)
            for d, ks in levels.items() for i, k in enumerate(ks)}

def add_tree(fig, col, edges, nodes):
    sfx = "" if col == 1 else str(col)
    xref, yref = f"x{sfx}", f"y{sfx}"
    pos = tree_layout(edges)
    for pk, ck, p, _d, s in edges:
        (x0, y0), (x1, y1) = pos[pk], pos[ck]
        color = "#c0392b" if not s else "#2b6cb0"
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref=xref, yref=yref, axref=xref,
                           ayref=yref, showarrow=True, arrowhead=3, arrowwidth=1.5 + p * 7.5,
                           arrowcolor=color, standoff=18, startstandoff=24, text="")
        fig.add_annotation(x=(x0 + x1) / 2, y=(y0 + y1) / 2, xref=xref, yref=yref,
                           showarrow=False, text=f"{p:.2f}", font=dict(size=10, color="#333"),
                           bgcolor="rgba(255,255,255,0.75)")
    keys = list(pos.keys())
    fig.add_trace(go.Scatter(
        x=[pos[k][0] for k in keys], y=[pos[k][1] for k in keys], mode="markers+text",
        text=[nodes[k][0] for k in keys], textposition="middle center",
        textfont=dict(size=13, color="#111"),
        marker=dict(color=["#f6c350" if k == "root" else ("#f6c0bb" if not nodes[k][1] else "#d9d9d9")
                           for k in keys],
                    size=[40 if k == "root" else 30 for k in keys], line=dict(width=1.5, color="#333")),
        hoverinfo="text", showlegend=False), row=1, col=col)
    fig.update_xaxes(visible=False, row=1, col=col)
    fig.update_yaxes(visible=False, row=1, col=col)

CONTEXTS = ["القطة بتاكل السمك", "الولد بياكل السمك"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=[f"السياق: {c}" for c in CONTEXTS])
for i, c in enumerate(CONTEXTS):
    e, n = build_prediction_tree(c, max_depth=2, top_k=2)
    add_tree(fig, i + 1, e, n)
fig.update_layout(height=460, margin=dict(l=20, r=20, t=80, b=20),
                  title_text="موديل بطبقتين على العربي · 2-layer model — prediction trees")
fig.show()

<div dir="rtl">

## ٧. الخلاصة والخطوة الجاية · Recap & handoff

- العمق فتح **التركيب**: رأس يستعمل ناتج رأس تاني.
- على مهمة صناعية نظيفة، شفنا الدائرة بوضوح: **رأس الكلمة-السابقة في L0** + **رأس الاستقراء في L1**، واتأكدنا إنها بتشتغل على تتابع جديد (تعلّم خوارزمية مش حفظ).
- رجعنا للعربي بموديل بطبقتين وأعدنا **شجرة التوقعات** على نفس السياقين.

ده **قمة** سُلّم الانتباه — ونفس النتيجة الأساسية في ورقة *A Mathematical Framework for Transformer Circuits* (٢٠٢١).

**المرحلة الجاية (٢د):** لحد دلوقتي كله انتباه بس. نضيف **الـ MLP** عشان نكمّل كتلة المحوّل — حساب لكل توكن لوحده (من غير ما يحرّك معلومة بين المواضع).

</div>

<div dir="ltr">

## 7. Recap & handoff

- Depth unlocked **composition**: one head using another's output.
- On a clean synthetic task we saw the circuit unmistakably: **prev-token head in L0** + **induction head in L1**, and confirmed it fires on a fresh sequence (an algorithm, not memorisation).
- We returned to Arabic with a 2-layer model and reused the **prediction tree** on the same two contexts.

This is the **capstone** of the attention ladder — the headline result of *A Mathematical Framework for Transformer Circuits* (2021).

**Next (2d):** so far it's all attention. We add the **MLP** to complete the transformer block — per-token computation that moves no information between positions.

</div>